# S06 · Cazador de errores (PRIMM) · tres programas, un error cada uno

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S06 · jueves 3 de diciembre de 2026 · 10:30–12:00 · Salón 3208

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. En Kaggle con P100 cambia `-arch=sm_75` por `-arch=sm_60`.
Anota la GPU asignada y la fecha (celda siguiente): el hardware de Colab cambia y el protocolo exige declararlo.

Sigue el `README.md` de la práctica guiada. **Escribe tus predicciones (P1–P8) antes de ejecutar** la sección "Ejecutar".
El tercer programa (MPI) está al final: Colab no trae MPI, pero se puede instalar en un minuto.

In [ ]:
!nvidia-smi
!nvcc --version | tail -1
!date

## Código
Las celdas siguientes escriben los archivos en el disco de Colab. Cuando corrijas un error (fase **Modificar**), edita la celda
`%%writefile` correspondiente, vuelve a ejecutarla **y** vuelve a ejecutar la celda de compilación.

In [ ]:
%%writefile caza2_guard_buggy.cu
/* caza2_guard_buggy.cu — Cazador de errores 2 (CUDA): y = a·x + b, un hilo por elemento.
 *
 * Qué hace cuando funciona: y[i] = 2·x[i] + 1 para i < n, con x[i] = i % 7.
 * Referencia: ./caza_cpu axpb 1000  →  checksum=6994.0
 *
 * Detector incluido: el programa reserva cap = grid·block elementos (múltiplo del bloque) y pone
 * y[n..cap) = -1 (centinelas). Si un hilo sobrante escribe ahí, se nota. En un programa real NO hay
 * centinelas: ese hilo escribe sobre otro arreglo o fuera de la reserva (a veces "illegal memory
 * access", a veces nada visible).
 *
 * Contiene UN error a propósito. Escribe tu predicción antes de correr:
 *   ./caza2_guard_buggy 1000 256     ¿cuántos hilos se lanzan? ¿cuántos sobran? ¿qué hacen?
 *   ./caza2_guard_buggy 1024 256     ¿y ahora?
 * Sin GPU: NUMBA_ENABLE_CUDASIM=1 python3 caza2_guard_numba.py 1000 256
 * Cuando lo corrijas (aquí y en el espejo Numba): make cazar
 */
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t e_ = (call);                                                           \
        if (e_ != cudaSuccess) {                                                           \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
            exit(1);                                                                       \
        }                                                                                  \
    } while (0)

__global__ void axpb(const float *x, float *y, float a, float b, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   /* índice global del hilo */
    y[i] = a * x[i] + b;                             /* un elemento por hilo */
}

static int cmp_float(const void *p, const void *q) {
    float a = *(const float *) p, b = *(const float *) q;
    return (a > b) - (a < b);
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : 1000;
    int block = argc > 2 ? atoi(argv[2]) : 256;
    if (n <= 0 || block <= 0 || block > 1024) {
        fprintf(stderr, "uso: %s [n>0] [block 1..1024]\n", argv[0]);
        return 1;
    }
    const float a = 2.0f, b = 1.0f;
    int grid = (n + block - 1) / block;              /* bloques: redondeo hacia arriba */
    int cap = grid * block;                          /* hilos lanzados = elementos reservados */

    /* Anfitrión: x válido en [0, n); relleno 0 en [n, cap). y arranca en -1 (centinela) en todo cap. */
    float *h_x = (float *) malloc(cap * sizeof(float));
    float *h_y = (float *) malloc(cap * sizeof(float));
    for (int i = 0; i < cap; i++) {
        h_x[i] = (i < n) ? (float) (i % 7) : 0.0f;
        h_y[i] = -1.0f;
    }

    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, cap * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_y, cap * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, cap * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, cap * sizeof(float), cudaMemcpyHostToDevice));

    /* Protocolo: 1 calentamiento + 5 repeticiones, eventos, mediana. Solo el kernel. */
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));
    float times[5];
    for (int r = -1; r < 5; r++) {
        CUDA_CHECK(cudaEventRecord(t0));
        axpb<<<grid, block>>>(d_x, d_y, a, b, n);
        CUDA_CHECK(cudaEventRecord(t1));
        CUDA_CHECK(cudaEventSynchronize(t1));
        if (r >= 0) CUDA_CHECK(cudaEventElapsedTime(&times[r], t0, t1));
    }
    cudaError_t launch_err = cudaGetLastError();        /* error de lanzamiento, si lo hubo */
    cudaError_t exec_err = cudaDeviceSynchronize();     /* error de ejecución (p. ej. acceso ilegal) */
    CUDA_CHECK(cudaMemcpy(h_y, d_y, cap * sizeof(float), cudaMemcpyDeviceToHost));

    /* Verificación: (1) los n válidos contra la CPU; (2) los centinelas siguen en -1. */
    int wrong = 0, altered = 0;
    double checksum = 0.0;
    for (int i = 0; i < n; i++) {
        float ref = a * h_x[i] + b;
        if (fabsf(h_y[i] - ref) > 1e-5f) wrong++;
        checksum += h_y[i];
    }
    for (int i = n; i < cap; i++)
        if (h_y[i] != -1.0f) altered++;
    qsort(times, 5, sizeof(float), cmp_float);

    printf("caza2: n=%d block=%d grid=%d hilos_lanzados=%d sobrantes=%d\n", n, block, grid, cap, cap - n);
    printf("cudaGetLastError=%s cudaDeviceSynchronize=%s\n",
           cudaGetErrorString(launch_err), cudaGetErrorString(exec_err));
    printf("checksum=%.1f elementos_incorrectos=%d centinelas_alterados=%d (de %d)\n",
           checksum, wrong, altered, cap - n);
    printf("t_kernel_ms=%.4f (mediana de 5, solo kernel)\n", times[2]);
    int ok = (wrong == 0 && altered == 0 && launch_err == cudaSuccess && exec_err == cudaSuccess);
    printf("verificacion=%s\n", ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaEventDestroy(t0));
    CUDA_CHECK(cudaEventDestroy(t1));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    free(h_x);
    free(h_y);
    return ok ? 0 : 2;
}


In [ ]:
%%writefile caza3_reduce_buggy.cu
/* caza3_reduce_buggy.cu — Cazador de errores 3 (CUDA): suma de un arreglo con reducción en árbol.
 *
 * Qué hace cuando funciona: cada bloque suma sus 256 elementos en memoria compartida (árbol de
 * log2(256) = 8 niveles, como en S03), deja una suma parcial por bloque y el anfitrión suma las parciales.
 * x[i] = (i % 10) + 1.  Referencia: ./caza_cpu suma 1000  →  suma=5500.0
 *
 * Contiene UN error a propósito. Escribe tu predicción antes de correr:
 *   ./caza3_reduce_buggy 1000        ¿suma correcta? ¿igual en cada corrida?
 *   for i in 1 2 3 4 5; do ./caza3_reduce_buggy 1000 | grep verificacion; done
 * Sin GPU: NUMBA_ENABLE_CUDASIM=1 python3 caza3_reduce_numba.py 1000
 * Cuando lo corrijas (aquí y en el espejo Numba): make cazar
 */
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define BLOCK 256      /* tamaño máximo de bloque: la memoria compartida se reserva con este tamaño */

#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t e_ = (call);                                                           \
        if (e_ != cudaSuccess) {                                                           \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
            exit(1);                                                                       \
        }                                                                                  \
    } while (0)

__global__ void block_sum(const float *x, float *partial, int n) {
    __shared__ float s[BLOCK];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;
    s[tid] = (i < n) ? x[i] : 0.0f;                  /* carga con guardia; los sobrantes aportan 0 */
    __syncthreads();                                 /* todo el bloque terminó de cargar */
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            s[tid] += s[tid + stride];               /* nivel del árbol: la mitad baja suma a la alta */
    }
    if (tid == 0) partial[blockIdx.x] = s[0];        /* suma parcial del bloque */
}

static int cmp_float(const void *p, const void *q) {
    float a = *(const float *) p, b = *(const float *) q;
    return (a > b) - (a < b);
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : 1000;
    int block = argc > 2 ? atoi(argv[2]) : BLOCK;
    if (n <= 0 || block <= 0 || block > BLOCK || (block & (block - 1)) != 0) {
        fprintf(stderr, "uso: %s [n>0] [block: potencia de 2, <= %d]\n", argv[0], BLOCK);
        return 1;
    }
    int grid = (n + block - 1) / block;

    float *h_x = (float *) malloc(n * sizeof(float));
    float *h_partial = (float *) malloc(grid * sizeof(float));
    double ref = 0.0;
    for (int i = 0; i < n; i++) { h_x[i] = (float) (i % 10) + 1.0f; ref += h_x[i]; }

    float *d_x, *d_partial;
    CUDA_CHECK(cudaMalloc(&d_x, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_partial, grid * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, n * sizeof(float), cudaMemcpyHostToDevice));

    /* Protocolo: 1 calentamiento + 5 repeticiones con eventos; solo el kernel. */
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));
    float times[5];
    for (int r = -1; r < 5; r++) {
        CUDA_CHECK(cudaEventRecord(t0));
        block_sum<<<grid, block>>>(d_x, d_partial, n);
        CUDA_CHECK(cudaEventRecord(t1));
        CUDA_CHECK(cudaEventSynchronize(t1));
        if (r >= 0) CUDA_CHECK(cudaEventElapsedTime(&times[r], t0, t1));
    }
    cudaError_t launch_err = cudaGetLastError();
    cudaError_t exec_err = cudaDeviceSynchronize();
    CUDA_CHECK(cudaMemcpy(h_partial, d_partial, grid * sizeof(float), cudaMemcpyDeviceToHost));

    /* El anfitrión termina la reducción (segunda etapa) y verifica contra la CPU. */
    double sum = 0.0;
    for (int b = 0; b < grid; b++) sum += h_partial[b];
    qsort(times, 5, sizeof(float), cmp_float);
    int ok = (fabs(sum - ref) <= 1e-6 * fabs(ref) && launch_err == cudaSuccess && exec_err == cudaSuccess);

    printf("caza3: n=%d block=%d grid=%d\n", n, block, grid);
    printf("cudaGetLastError=%s cudaDeviceSynchronize=%s\n",
           cudaGetErrorString(launch_err), cudaGetErrorString(exec_err));
    if (grid <= 8) {
        printf("parciales_gpu=");
        for (int b = 0; b < grid; b++) printf("%s%.1f", b ? "," : "", h_partial[b]);
        printf("\n");
    }
    printf("suma_gpu=%.1f suma_cpu=%.1f\n", sum, ref);
    printf("t_kernel_ms=%.4f (mediana de 5, solo kernel)\n", times[2]);
    printf("verificacion=%s\n", ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaEventDestroy(t0));
    CUDA_CHECK(cudaEventDestroy(t1));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_partial));
    free(h_x);
    free(h_partial);
    return ok ? 0 : 2;
}


In [ ]:
%%writefile caza_cpu.c
/* caza_cpu.c — Referencia secuencial de los cálculos del cazador de errores y del ejemplo resuelto.
 *
 * Uso:  ./caza_cpu media [n]   media de x[i] = (i % 7) + 0.5          (caza1, MPI)
 *       ./caza_cpu axpb  [n]   checksum de y = 2·x + 1, x[i] = i % 7  (caza2, CUDA)
 *       ./caza_cpu suma  [n]   suma de x[i] = (i % 10) + 1            (caza3, CUDA)
 *       ./caza_cpu grid  [n]   checksum de y = 0.5·x, x[i] = (i%7)+1  (ejemplo resuelto)
 *       ./caza_cpu todo  [n]   las cuatro
 * Imprime el valor de referencia y el tiempo (mediana de 5 repeticiones tras 1 calentamiento),
 * que sirve como T(1) para el speedup del kernel en la celda de medición del notebook.
 * Compila con: gcc -std=c11 -O2 -Wall -Wextra caza_cpu.c -o caza_cpu
 */
#define _POSIX_C_SOURCE 199309L
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define REPS 5

static double now_s(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec * 1e-9;
}

static double median5(double *t) {              /* ordena 5 valores y regresa el del centro */
    for (int i = 1; i < REPS; i++)
        for (int j = i; j > 0 && t[j - 1] > t[j]; j--) {
            double tmp = t[j]; t[j] = t[j - 1]; t[j - 1] = tmp;
        }
    return t[REPS / 2];
}

/* Cada cálculo recibe un arreglo de trabajo ya reservado y regresa su resultado escalar. */
static double calc_media(const float *x, float *y, int n) {
    (void) y;
    double s = 0.0;
    for (int i = 0; i < n; i++) s += x[i];
    return s / n;
}
static double calc_axpb(const float *x, float *y, int n) {
    double s = 0.0;
    for (int i = 0; i < n; i++) { y[i] = 2.0f * x[i] + 1.0f; s += y[i]; }
    return s;
}
static double calc_suma(const float *x, float *y, int n) {
    (void) y;
    double s = 0.0;
    for (int i = 0; i < n; i++) s += x[i];
    return s;
}
static double calc_grid(const float *x, float *y, int n) {
    double s = 0.0;
    for (int i = 0; i < n; i++) { y[i] = 0.5f * x[i]; s += y[i]; }
    return s;
}

typedef struct {
    const char *name;
    const char *label;      /* etiqueta del resultado en la salida */
    int decimals;
    float (*init)(int i);
    double (*calc)(const float *, float *, int);
} job_t;

static float init_media(int i) { return (float) (i % 7) + 0.5f; }
static float init_axpb(int i)  { return (float) (i % 7); }
static float init_suma(int i)  { return (float) (i % 10) + 1.0f; }
static float init_grid(int i)  { return (float) (i % 7) + 1.0f; }

static const job_t JOBS[] = {
    { "media", "media",    6, init_media, calc_media },
    { "axpb",  "checksum", 1, init_axpb,  calc_axpb  },
    { "suma",  "suma",     1, init_suma,  calc_suma  },
    { "grid",  "checksum", 1, init_grid,  calc_grid  },
};
#define NJOBS ((int) (sizeof JOBS / sizeof JOBS[0]))

static void run_job(const job_t *job, int n) {
    float *x = malloc(n * sizeof(float));
    float *y = malloc(n * sizeof(float));
    if (!x || !y) { fprintf(stderr, "sin memoria para n=%d\n", n); exit(1); }
    for (int i = 0; i < n; i++) x[i] = job->init(i);

    double result = 0.0, times[REPS];
    for (int r = -1; r < REPS; r++) {            /* r = -1: calentamiento, no se cuenta */
        double t0 = now_s();
        result = job->calc(x, y, n);
        double t1 = now_s();
        if (r >= 0) times[r] = t1 - t0;
    }
    printf("caza_cpu %s: n=%d %s=%.*f t_cpu_ms=%.4f (mediana de %d)\n",
           job->name, n, job->label, job->decimals, result, median5(times) * 1e3, REPS);
    free(x);
    free(y);
}

int main(int argc, char **argv) {
    const char *what = argc > 1 ? argv[1] : "todo";
    int n = argc > 2 ? atoi(argv[2]) : 1000;
    if (n <= 0) { fprintf(stderr, "uso: %s media|axpb|suma|grid|todo [n>0]\n", argv[0]); return 1; }
    int done = 0;
    for (int j = 0; j < NJOBS; j++) {
        if (strcmp(what, "todo") == 0 || strcmp(what, JOBS[j].name) == 0) { run_job(&JOBS[j], n); done++; }
    }
    if (!done) { fprintf(stderr, "uso: %s media|axpb|suma|grid|todo [n>0]\n", argv[0]); return 1; }
    return 0;
}


In [ ]:
%%writefile caza2_guard_numba.py
#!/usr/bin/env python3
"""caza2_guard_numba.py — Espejo en Numba CUDA de caza2_guard_buggy.cu (contiene el MISMO error).

Uso:
    python3 caza2_guard_numba.py [n] [block]
Sin GPU (simulador en CPU; solo correctitud):
    NUMBA_ENABLE_CUDASIM=1 python3 caza2_guard_numba.py 1000 256

Misma reserva con capacidad cap = grid·block, mismos centinelas (-1) en y[n..cap), misma verificación
y mismo formato de salida que la versión en C. Cuando corrijas el .cu, haz el mismo cambio aquí:
`make cazar` usa este archivo para verificarte sin GPU.
"""
import os
import sys
import time

import numpy as np
from numba import cuda

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 1 if CUDASIM else 5     # en el simulador el tiempo no significa nada: una sola ejecución


@cuda.jit
def axpb(x, y, a, b, n):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x   # índice global del hilo
    y[i] = a * x[i] + b                                        # un elemento por hilo


def median(values):
    return sorted(values)[len(values) // 2]


def main():
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 1000
    block = int(sys.argv[2]) if len(sys.argv) > 2 else 256
    if n <= 0 or block <= 0 or block > 1024:
        print("uso: caza2_guard_numba.py [n>0] [block 1..1024]")
        sys.exit(1)
    if CUDASIM and n > 20000:
        print("aviso: CUDASIM con n > 20000 tarda minutos", file=sys.stderr)
    a, b = np.float32(2.0), np.float32(1.0)
    grid = (n + block - 1) // block          # redondeo hacia arriba
    cap = grid * block                       # hilos lanzados = elementos reservados

    # Anfitrión: x válido en [0, n), relleno 0 después; y arranca en -1 (centinela) en todo cap.
    h_x = np.zeros(cap, dtype=np.float32)
    h_x[:n] = (np.arange(n) % 7).astype(np.float32)
    h_y = np.full(cap, -1.0, dtype=np.float32)

    d_x = cuda.to_device(h_x)                # cudaMalloc + cudaMemcpy H2D
    d_y = cuda.to_device(h_y)

    # Protocolo: 1 calentamiento (incluye el JIT en GPU real) + 5 repeticiones; solo el kernel.
    if not CUDASIM:                          # calentamiento (incluye el JIT en GPU real)
        axpb[grid, block](d_x, d_y, a, b, n)
        cuda.synchronize()
    times_ms = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        axpb[grid, block](d_x, d_y, a, b, n)
        cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1e3)
    d_y.copy_to_host(h_y)                    # cudaMemcpy D2H

    # Verificación: (1) los n válidos contra NumPy; (2) los centinelas siguen en -1.
    ref = a * h_x[:n] + b
    wrong = int(np.sum(np.abs(h_y[:n] - ref) > 1e-5))
    altered = int(np.sum(h_y[n:] != -1.0))
    checksum = float(np.sum(h_y[:n], dtype=np.float64))
    ok = wrong == 0 and altered == 0
    modo = "CUDASIM" if CUDASIM else "GPU"

    print(f"caza2: n={n} block={block} grid={grid} hilos_lanzados={cap} sobrantes={cap - n} modo={modo}")
    print(f"checksum={checksum:.1f} elementos_incorrectos={wrong} centinelas_alterados={altered} (de {cap - n})")
    print(f"t_kernel_ms={median(times_ms):.4f} (mediana de {REPS}, solo kernel)")
    print(f"verificacion={'OK' if ok else 'FALLA'}")
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile caza3_reduce_numba.py
#!/usr/bin/env python3
"""caza3_reduce_numba.py — Espejo en Numba CUDA de caza3_reduce_buggy.cu (contiene el MISMO error).

Uso:
    python3 caza3_reduce_numba.py [n] [block: potencia de 2, <= 256]
Sin GPU (simulador en CPU; solo correctitud):
    NUMBA_ENABLE_CUDASIM=1 python3 caza3_reduce_numba.py 1000

El simulador ejecuta cada hilo del bloque como un hilo de Python y solo los coordina en
cuda.syncthreads(): si falta una barrera, el orden de ejecución es arbitrario y el resultado también.
Cuando corrijas el .cu, haz el mismo cambio aquí: `make cazar` usa este archivo para verificarte sin GPU.
"""
import os
import sys
import time

import numpy as np
from numba import cuda, float32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 1 if CUDASIM else 5     # en el simulador el tiempo no significa nada: una sola ejecución
BLOCK = 256     # tamaño máximo de bloque: la memoria compartida se reserva con este tamaño


@cuda.jit
def block_sum(x, partial, n):
    s = cuda.shared.array(BLOCK, float32)
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid
    s[tid] = x[i] if i < n else 0.0           # carga con guardia; los sobrantes aportan 0
    cuda.syncthreads()                        # todo el bloque terminó de cargar
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            s[tid] += s[tid + stride]         # nivel del árbol: la mitad baja suma a la alta
        stride //= 2
    if tid == 0:
        partial[cuda.blockIdx.x] = s[0]       # suma parcial del bloque


def median(values):
    return sorted(values)[len(values) // 2]


def main():
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 1000
    block = int(sys.argv[2]) if len(sys.argv) > 2 else BLOCK
    if n <= 0 or block <= 0 or block > BLOCK or (block & (block - 1)) != 0:
        print(f"uso: caza3_reduce_numba.py [n>0] [block: potencia de 2, <= {BLOCK}]")
        sys.exit(1)
    if CUDASIM and n > 20000:
        print("aviso: CUDASIM con n > 20000 tarda minutos", file=sys.stderr)
    grid = (n + block - 1) // block

    h_x = (np.arange(n) % 10 + 1).astype(np.float32)
    ref = float(np.sum(h_x, dtype=np.float64))
    h_partial = np.zeros(grid, dtype=np.float32)

    d_x = cuda.to_device(h_x)
    d_partial = cuda.to_device(h_partial)

    # Protocolo: 1 calentamiento + 5 repeticiones; solo el kernel.
    if not CUDASIM:                          # calentamiento (incluye el JIT en GPU real)
        block_sum[grid, block](d_x, d_partial, n)
        cuda.synchronize()
    times_ms = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        block_sum[grid, block](d_x, d_partial, n)
        cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1e3)
    d_partial.copy_to_host(h_partial)

    # El anfitrión termina la reducción (segunda etapa) y verifica contra NumPy.
    total = float(np.sum(h_partial, dtype=np.float64))
    ok = abs(total - ref) <= 1e-6 * abs(ref)
    modo = "CUDASIM" if CUDASIM else "GPU"

    print(f"caza3: n={n} block={block} grid={grid} modo={modo}")
    if grid <= 8:
        print("parciales_gpu=" + ",".join(f"{p:.1f}" for p in h_partial))
    print(f"suma_gpu={total:.1f} suma_cpu={ref:.1f}")
    print(f"t_kernel_ms={median(times_ms):.4f} (mediana de {REPS}, solo kernel)")
    print(f"verificacion={'OK' if ok else 'FALLA'}")
    sys.exit(0 if ok else 2)


if __name__ == "__main__":
    main()


## Compilar

In [ ]:
!nvcc -O2 -arch=sm_75 caza2_guard_buggy.cu -o caza2_guard_buggy && echo 'caza2 listo'
!nvcc -O2 -arch=sm_75 caza3_reduce_buggy.cu -o caza3_reduce_buggy && echo 'caza3 listo'
!gcc -std=c11 -O2 -Wall -Wextra caza_cpu.c -o caza_cpu && echo 'caza_cpu listo'

## Ejecutar (PRIMM · R)
Primero la referencia en CPU (los valores que deben salir), después cada programa con error. Compara con P1–P8 del README.
Fíjate en tres cosas por programa: `cudaGetLastError`, el resultado contra la CPU y el código de salida.

In [ ]:
!./caza_cpu todo 1000

In [ ]:
!./caza2_guard_buggy 1000 256; echo "codigo de salida: $?"

In [ ]:
!./caza2_guard_buggy 1024 256; echo "codigo de salida: $?"

In [ ]:
!./caza3_reduce_buggy 1000; echo "codigo de salida: $?"

In [ ]:
# Cinco corridas seguidas: ¿sale lo mismo cada vez?
!for i in 1 2 3 4 5; do ./caza3_reduce_buggy 1000 | grep -E 'suma_gpu|verificacion' | tr '\n' ' '; echo; done

## Investigar (PRIMM · I)
El simulador de Numba corre el **mismo** kernel en la CPU, con cada hilo del bloque como un hilo de Python que solo se coordina
en `syncthreads()`. No tiene GPU y no mide nada útil, pero reproduce (o no) el error con otro orden de ejecución.
Preguntas del README, sección Investigar: ¿por qué la GPU y el simulador pueden dar números distintos para caza3 y ninguno de los dos "tiene razón"?

In [ ]:
!NUMBA_ENABLE_CUDASIM=1 python3 caza2_guard_numba.py 1000 256; echo "codigo de salida: $?"

In [ ]:
!NUMBA_ENABLE_CUDASIM=1 python3 caza3_reduce_numba.py 1000; echo "codigo de salida: $?"

In [ ]:
# Bloque de 32 hilos (un solo warp): ¿cambia algo en la GPU? ¿y en el simulador? ¿Cuál de los dos está bien? (ninguno: es comportamiento indefinido)
!for i in 1 2 3; do ./caza3_reduce_buggy 1000 32 | grep -E 'suma_gpu|verificacion' | tr '\n' ' '; echo; done
!NUMBA_ENABLE_CUDASIM=1 python3 caza3_reduce_numba.py 1000 32 | grep -E 'suma_gpu|verificacion'

## Modificar (PRIMM · M)
Corrige **una línea** en cada programa (en el `.cu` y en su espejo `_numba.py`): edita las celdas `%%writefile`, ejecútalas,
recompila y corre la celda de verificación. Es el equivalente de `make cazar` del repositorio: pasa solo cuando los dos están corregidos.

In [ ]:
import subprocess

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, p.stdout + p.stderr

casos = [
    ("caza2 GPU     n=1000 b=256", "./caza2_guard_buggy 1000 256"),
    ("caza2 GPU     n=37   b=8",   "./caza2_guard_buggy 37 8"),
    ("caza2 CUDASIM n=1000 b=256", "NUMBA_ENABLE_CUDASIM=1 python3 caza2_guard_numba.py 1000 256"),
    ("caza3 GPU     n=1000 b=256", "./caza3_reduce_buggy 1000"),
    ("caza3 GPU     n=1000 b=256 (2a vez)", "./caza3_reduce_buggy 1000"),
    ("caza3 GPU     n=1000 b=64",  "./caza3_reduce_buggy 1000 64"),
    ("caza3 CUDASIM n=1000 b=256", "NUMBA_ENABLE_CUDASIM=1 python3 caza3_reduce_numba.py 1000"),
]
fallas = 0
for nombre, cmd in casos:
    rc, out = run(cmd)
    ok = rc == 0 and "verificacion=OK" in out
    fallas += 0 if ok else 1
    print(f"{'PASS' if ok else 'FAIL'}  {nombre}")
print("\nbichos cazados: " + ("2/2 (CUDA)" if fallas == 0 else f"todavía no: {fallas} casos fallan"))


## Medir (protocolo del curso) — solo con los programas ya corregidos
Cada programa hace 1 calentamiento + 5 repeticiones con `cudaEvent` y reporta la **mediana** del tiempo **solo del kernel**
(estos programas no miden las transferencias: por eso el *speedup* de esta tabla es "de kernel", no el del programa; en tu V3
reportas los dos). `T(1)` es `caza_cpu` con `-O2` en la CPU de la máquina virtual de Colab.

In [ ]:
import re, subprocess, datetime

def campo(texto, clave):
    m = re.search(clave + r"=([-0-9.]+)", texto)
    return float(m.group(1)) if m else float("nan")

filas = []
for n in [1 << 16, 1 << 20, 1 << 24]:
    for prog, tarea, cpu_cmd in [("caza2_guard_buggy", "axpb", "axpb"), ("caza3_reduce_buggy", "suma", "suma")]:
        gpu = subprocess.run(f"./{prog} {n} 256", shell=True, capture_output=True, text=True).stdout
        cpu = subprocess.run(f"./caza_cpu {cpu_cmd} {n}", shell=True, capture_output=True, text=True).stdout
        t_k, t_c = campo(gpu, "t_kernel_ms"), campo(cpu, "t_cpu_ms")
        ver = "OK" if "verificacion=OK" in gpu else "FALLA"
        filas.append([prog, n, 256, ver, t_k, t_c, t_c / t_k if t_k > 0 else float("nan")])
        print(f"{prog:18s} n={n:>9d} verif={ver:5s} t_kernel_ms={t_k:8.4f} t_cpu_ms={t_c:9.4f} speedup_kernel={t_c / t_k:7.1f}")
print("\nUna fila con FALLA no tiene speedup: primero corrige, luego mide.")


## Ejemplo resuelto y ítems CUDA del examen de práctica — comprueba tu predicción
`ejemplo_grid.cu` es la parte B de `ejemplo-resuelto.md` (el error está en el anfitrión: `grid = n / block`); se compila dos veces,
con y sin `-DCORREGIDO`. `item_II2_reverse.cu` es el ítem II.2 del examen de práctica (predecir la salida de un kernel) e
`item_III2_hist.cu` el ítem III.2 (encontrar y corregir el error). **Escribe tu respuesta en el examen antes de ejecutar estas celdas.**

In [ ]:
%%writefile ejemplo_grid.cu
/* ejemplo_grid.cu — Ejemplo resuelto (parte B): cómo se encuentra y corrige un error en un kernel.
 *
 * y = 0.5·x con x[i] = (i % 7) + 1 y n = 1000. El kernel tiene guardia. El error está en el ANFITRIÓN:
 * el número de bloques se calcula con división entera (n / block) y los últimos n % block elementos
 * nunca se procesan. Referencia: ./caza_cpu grid 1000 → checksum=1998.5
 *   nvcc -O2 -arch=sm_75 ejemplo_grid.cu -o ejemplo_grid && ./ejemplo_grid 1000 256
 *   nvcc -O2 -arch=sm_75 -DCORREGIDO ejemplo_grid.cu -o ejemplo_grid_ok && ./ejemplo_grid_ok 1000 256
 * Sin GPU: NUMBA_ENABLE_CUDASIM=1 python3 ejemplo_grid_numba.py 1000 256 [--corregido]
 */
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t e_ = (call);                                                           \
        if (e_ != cudaSuccess) {                                                           \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
            exit(1);                                                                       \
        }                                                                                  \
    } while (0)

__global__ void scale(const float *x, float *y, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = 0.5f * x[i];                   /* guardia presente: este kernel está bien */
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : 1000;
    int block = argc > 2 ? atoi(argv[2]) : 256;
    if (n <= 0 || block <= 0 || block > 1024) {
        fprintf(stderr, "uso: %s [n>0] [block 1..1024]\n", argv[0]);
        return 1;
    }
#ifdef CORREGIDO
    int grid = (n + block - 1) / block;              /* redondeo hacia arriba: cubre los n elementos */
#else
    int grid = n / block;                            /* división entera: ¿cuántos elementos quedan sin hilo? */
#endif
    float *h_x = (float *) malloc(n * sizeof(float));
    float *h_y = (float *) malloc(n * sizeof(float));
    for (int i = 0; i < n; i++) { h_x[i] = (float) (i % 7) + 1.0f; h_y[i] = 0.0f; }

    float *d_x, *d_y;
    CUDA_CHECK(cudaMalloc(&d_x, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_y, n * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, n * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, n * sizeof(float), cudaMemcpyHostToDevice));

    scale<<<grid, block>>>(d_x, d_y, n);
    cudaError_t launch_err = cudaGetLastError();
    cudaError_t exec_err = cudaDeviceSynchronize();
    CUDA_CHECK(cudaMemcpy(h_y, d_y, n * sizeof(float), cudaMemcpyDeviceToHost));

    int unprocessed = 0, wrong = 0;
    double checksum = 0.0;
    for (int i = 0; i < n; i++) {
        float ref = 0.5f * h_x[i];                   /* ref >= 0.5: un 0 en y significa "nadie lo tocó" */
        if (h_y[i] == 0.0f) unprocessed++;
        else if (fabsf(h_y[i] - ref) > 1e-5f) wrong++;
        checksum += h_y[i];
    }
    int ok = (unprocessed == 0 && wrong == 0 && launch_err == cudaSuccess && exec_err == cudaSuccess);
    printf("ejemplo_grid: n=%d block=%d grid=%d hilos_lanzados=%d\n", n, block, grid, grid * block);
    printf("cudaGetLastError=%s cudaDeviceSynchronize=%s\n",
           cudaGetErrorString(launch_err), cudaGetErrorString(exec_err));
    printf("checksum=%.1f elementos_incorrectos=%d elementos_sin_procesar=%d\n", checksum, wrong, unprocessed);
    printf("verificacion=%s\n", ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    free(h_x);
    free(h_y);
    return ok ? 0 : 2;
}


In [ ]:
%%writefile item_II2_reverse.cu
/* item_II2_reverse.cu — Examen de práctica U4–U5, sección II, ítem 2 (predecir la salida de un kernel).
 *   nvcc -O2 -arch=sm_75 item_II2_reverse.cu -o item_II2_reverse && ./item_II2_reverse
 * Sin GPU: NUMBA_ENABLE_CUDASIM=1 python3 item_II2_reverse_numba.py
 */
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t e_ = (call);                                                           \
        if (e_ != cudaSuccess) {                                                           \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
            exit(1);                                                                       \
        }                                                                                  \
    } while (0)

__global__ void reverse_in_block(int *data, int n) {
    __shared__ int s[4];
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) s[threadIdx.x] = data[i];
    __syncthreads();
    int j = blockDim.x - 1 - threadIdx.x;            /* posición espejo dentro del bloque */
    int src = blockIdx.x * blockDim.x + j;           /* índice global de esa posición */
    if (i < n && src < n) data[i] = s[j];
}

int main(void) {
    const int n = 10;
    int h[n];
    for (int i = 0; i < n; i++) h[i] = i;
    int *d;
    CUDA_CHECK(cudaMalloc(&d, n * sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d, h, n * sizeof(int), cudaMemcpyHostToDevice));
    reverse_in_block<<<3, 4>>>(d, n);
    CUDA_CHECK(cudaGetLastError());                  /* lanzamiento válido */
    CUDA_CHECK(cudaDeviceSynchronize());             /* ejecución sin error */
    CUDA_CHECK(cudaMemcpy(h, d, n * sizeof(int), cudaMemcpyDeviceToHost));
    for (int i = 0; i < n; i++) printf("%d%s", h[i], i < n - 1 ? " " : "\n");
    CUDA_CHECK(cudaFree(d));
    return 0;
}


In [ ]:
%%writefile item_III2_hist.cu
/* item_III2_hist.cu — Examen de práctica U4–U5, sección III, ítem 2 (encontrar y corregir el error).
 *
 * Histograma de 8 bins: v[i] = i % 8, así que cada bin debe valer exactamente n / 8. El kernel tiene
 * índice global y guardia correctos; el error está en cómo se actualiza el contador compartido.
 *   nvcc -O2 -arch=sm_75 item_III2_hist.cu -o item_III2_hist && ./item_III2_hist 1048576
 *   nvcc -O2 -arch=sm_75 -DCORREGIDO item_III2_hist.cu -o item_III2_hist_ok && ./item_III2_hist_ok 1048576
 * Sin GPU: NUMBA_ENABLE_CUDASIM=1 python3 item_III2_hist_numba.py 1000 [--corregido]
 */
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define BINS 8

#define CUDA_CHECK(call)                                                                   \
    do {                                                                                   \
        cudaError_t e_ = (call);                                                           \
        if (e_ != cudaSuccess) {                                                           \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e_)); \
            exit(1);                                                                       \
        }                                                                                  \
    } while (0)

__global__ void hist8(const unsigned char *v, unsigned int *hist, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
#ifdef CORREGIDO
        atomicAdd(&hist[v[i]], 1u);         /* lectura-suma-escritura indivisible */
#else
        hist[v[i]] += 1;                    /* lectura, suma y escritura por separado: otros hilos se cuelan */
#endif
    }
}

int main(int argc, char **argv) {
    int n = argc > 1 ? atoi(argv[1]) : (1 << 20);
    int block = 256;
    if (n <= 0) { fprintf(stderr, "uso: %s [n>0]\n", argv[0]); return 1; }
    int grid = (n + block - 1) / block;

    unsigned char *h_v = (unsigned char *) malloc(n);
    unsigned int h_hist[BINS] = {0}, ref[BINS] = {0};
    for (int i = 0; i < n; i++) { h_v[i] = (unsigned char) (i % BINS); ref[h_v[i]]++; }

    unsigned char *d_v;
    unsigned int *d_hist;
    CUDA_CHECK(cudaMalloc(&d_v, n));
    CUDA_CHECK(cudaMalloc(&d_hist, BINS * sizeof(unsigned int)));
    CUDA_CHECK(cudaMemcpy(d_v, h_v, n, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_hist, 0, BINS * sizeof(unsigned int)));

    hist8<<<grid, block>>>(d_v, d_hist, n);
    cudaError_t launch_err = cudaGetLastError();
    cudaError_t exec_err = cudaDeviceSynchronize();
    CUDA_CHECK(cudaMemcpy(h_hist, d_hist, BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));

    unsigned long total = 0;
    int wrong = 0;
    printf("hist_gpu=[");
    for (int b = 0; b < BINS; b++) {
        printf("%u%s", h_hist[b], b < BINS - 1 ? "," : "]\n");
        total += h_hist[b];
        if (h_hist[b] != ref[b]) wrong++;
    }
    printf("hist_cpu=[");
    for (int b = 0; b < BINS; b++) printf("%u%s", ref[b], b < BINS - 1 ? "," : "]\n");
    printf("cudaGetLastError=%s cudaDeviceSynchronize=%s\n",
           cudaGetErrorString(launch_err), cudaGetErrorString(exec_err));
    int ok = (wrong == 0 && launch_err == cudaSuccess && exec_err == cudaSuccess);
    printf("n=%d total_gpu=%lu bins_incorrectos=%d verificacion=%s\n", n, total, wrong, ok ? "OK" : "FALLA");

    CUDA_CHECK(cudaFree(d_v));
    CUDA_CHECK(cudaFree(d_hist));
    free(h_v);
    return ok ? 0 : 2;
}


In [ ]:
%%writefile item_III2_hist_numba.py
#!/usr/bin/env python3
"""item_III2_hist_numba.py — Espejo en Numba del ítem III.2 del examen de práctica (histograma de 8 bins).

Uso:
    NUMBA_ENABLE_CUDASIM=1 python3 item_III2_hist_numba.py [n] [--corregido]
Sin --corregido reproduce el kernel con el error (hist[b] += 1 sin atómica); con --corregido usa
cuda.atomic.add. Aviso: el simulador ejecuta los hilos de un bloque como hilos de Python y rara vez
intercala la lectura y la escritura de dos hilos sobre el mismo contador, así que la carrera casi nunca
se manifiesta aquí; en la GPU, con n = 2^20, se pierden casi todas las cuentas. Una carrera que "no se
ve" en una plataforma no está corregida: se razona sobre los accesos.
"""
import os
import sys

import numpy as np
from numba import cuda

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
BINS = 8


@cuda.jit
def hist8_buggy(v, hist, n):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        hist[v[i]] += 1                       # lectura, suma y escritura por separado


@cuda.jit
def hist8_fixed(v, hist, n):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        cuda.atomic.add(hist, v[i], 1)        # indivisible


def main():
    flags = [a for a in sys.argv[1:] if a.startswith("--")]
    args = [a for a in sys.argv[1:] if not a.startswith("--")]
    n = int(args[0]) if args else 1000
    if n <= 0:
        print("uso: item_III2_hist_numba.py [n>0] [--corregido]")
        sys.exit(1)
    corregido = "--corregido" in flags
    block = 256
    grid = (n + block - 1) // block

    h_v = (np.arange(n) % BINS).astype(np.uint8)
    ref = np.bincount(h_v, minlength=BINS).astype(np.uint32)
    d_v = cuda.to_device(h_v)
    d_hist = cuda.to_device(np.zeros(BINS, dtype=np.uint32))
    kernel = hist8_fixed if corregido else hist8_buggy
    kernel[grid, block](d_v, d_hist, n)
    cuda.synchronize()
    h_hist = d_hist.copy_to_host()

    wrong = int(np.sum(h_hist != ref))
    modo = "CUDASIM" if CUDASIM else "GPU"
    print(f"hist_gpu=[{','.join(str(int(x)) for x in h_hist)}] modo={modo}{' (corregido)' if corregido else ''}")
    print(f"hist_cpu=[{','.join(str(int(x)) for x in ref)}]")
    print(f"n={n} total_gpu={int(h_hist.sum())} bins_incorrectos={wrong} verificacion={'OK' if wrong == 0 else 'FALLA'}")
    sys.exit(0 if wrong == 0 else 2)


if __name__ == "__main__":
    main()


In [ ]:
!nvcc -O2 -arch=sm_75 ejemplo_grid.cu -o ejemplo_grid && nvcc -O2 -arch=sm_75 -DCORREGIDO ejemplo_grid.cu -o ejemplo_grid_ok && echo 'ejemplo_grid listo'
!nvcc -O2 -arch=sm_75 item_II2_reverse.cu -o item_II2_reverse && echo 'II.2 listo'
!nvcc -O2 -arch=sm_75 item_III2_hist.cu -o item_III2_hist && nvcc -O2 -arch=sm_75 -DCORREGIDO item_III2_hist.cu -o item_III2_hist_ok && echo 'III.2 listo'

In [ ]:
# Ejemplo resuelto, parte B: el mismo kernel, dos anfitriones. ¿Hubo error de CUDA en el primero?
!./ejemplo_grid 1000 256; echo "codigo de salida: $?"
!./ejemplo_grid_ok 1000 256; echo "codigo de salida: $?"

In [ ]:
# Ítem II.2: la clave del examen dice 3 2 1 0 7 6 5 4 8 9. ¿Por qué el último bloque no se invierte?
!./item_II2_reverse

In [ ]:
# Ítem III.2: con el error, en la GPU se pierden casi todas las cuentas (total_gpu << n); corregido, exacto.
!./item_III2_hist 1048576; echo "codigo de salida: $?"
!./item_III2_hist_ok 1048576; echo "codigo de salida: $?"
# El simulador casi nunca intercala dos hilos sobre el mismo contador: la carrera se razona, no se observa aquí.
!NUMBA_ENABLE_CUDASIM=1 python3 item_III2_hist_numba.py 1000

## Opcional en Colab · el error de MPI (caza1)
Colab no trae MPI. Esta celda lo instala (≈ 1 minuto) y corre `caza1_bcast_buggy` con 4 procesos y un `timeout` de 5 s.
Código de salida 124 = `timeout` lo mató = interbloqueo. Con `-np 1` "funciona". Corrige la celda `%%writefile`, recompila y repite.

In [ ]:
!apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null 2>&1 && echo 'OpenMPI instalado'

In [ ]:
%%writefile caza1_bcast_buggy.c
/* caza1_bcast_buggy.c — Cazador de errores 1 (MPI): media de un arreglo repartido.
 *
 * Qué hace cuando funciona: el rango 0 toma n de la línea de comandos y lo anuncia a todos;
 * genera x[i] = (i % 7) + 0.5; reparte bloques con MPI_Scatterv (reparto con residuo de U4 S04);
 * cada rango suma su bloque; MPI_Reduce junta las sumas y el rango 0 imprime la media.
 * Referencia: ./caza_cpu media 1000  →  media=3.497000
 *
 * Contiene UN error a propósito. Escribe tu predicción antes de correrlo:
 *   timeout 5 mpirun --oversubscribe -np 4 ./caza1_bcast_buggy 1000 ; echo "codigo: $?"
 *   (código 124 = timeout lo mató = interbloqueo)
 *   timeout 5 mpirun --oversubscribe -np 1 ./caza1_bcast_buggy 1000
 * Cuando lo corrijas: make cazar
 */
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

/* Reparto por bloques con residuo: los primeros n % p rangos reciben una celda más. */
static void block_partition(int n, int p, int *counts, int *displs) {
    int offset = 0;
    for (int r = 0; r < p; r++) {
        counts[r] = n / p + (r < n % p ? 1 : 0);
        displs[r] = offset;
        offset += counts[r];
    }
}

int main(int argc, char **argv) {
    int rank, size, n = 0;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    /* 1. Solo el rango 0 conoce n: hay que anunciarlo a los demás. */
    if (rank == 0) {
        n = (argc > 1) ? atoi(argv[1]) : 1000;
        if (n <= 0) {
            fprintf(stderr, "uso: %s [n>0]\n", argv[0]);
            MPI_Abort(MPI_COMM_WORLD, 1);
        }
        MPI_Bcast(&n, 1, MPI_INT, 0, MPI_COMM_WORLD);                        /* la raíz difunde n   */
    } else {
        MPI_Recv(&n, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);   /* los demás lo reciben */
    }

    /* 2. Reparto de x entre los procesos. */
    int *counts = malloc(size * sizeof(int));
    int *displs = malloc(size * sizeof(int));
    block_partition(n, size, counts, displs);
    double *x = NULL;
    if (rank == 0) {
        x = malloc(n * sizeof(double));
        for (int i = 0; i < n; i++) x[i] = (i % 7) + 0.5;
    }
    double *local = malloc(counts[rank] * sizeof(double));
    MPI_Scatterv(x, counts, displs, MPI_DOUBLE, local, counts[rank], MPI_DOUBLE, 0, MPI_COMM_WORLD);

    /* 3. Suma local y reducción en la raíz. */
    double local_sum = 0.0, total = 0.0;
    for (int i = 0; i < counts[rank]; i++) local_sum += local[i];
    MPI_Reduce(&local_sum, &total, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) printf("caza1: n=%d procesos=%d media=%.6f\n", n, size, total / n);

    free(local);
    free(x);
    free(counts);
    free(displs);
    MPI_Finalize();
    return 0;
}


In [ ]:
!mpicc -std=c11 -O2 -Wall -Wextra caza1_bcast_buggy.c -o caza1_bcast_buggy && echo 'caza1 listo'
!./caza_cpu media 1000
!timeout 5 mpirun --allow-run-as-root --oversubscribe -np 4 ./caza1_bcast_buggy 1000; echo "codigo de salida: $? (124 = interbloqueo)"
!timeout 5 mpirun --allow-run-as-root --oversubscribe -np 1 ./caza1_bcast_buggy 1000; echo "codigo de salida: $?"

### Ítems MPI del examen de práctica (II.1 predecir la salida · III.1 encontrar el error · IV.a completar)
Comprueba tu respuesta **después** de escribirla. III.1 se compila dos veces: con el error y con `-DCORREGIDO`.

In [ ]:
%%writefile item_II1_mpi.c
/* item_II1_mpi.c — Examen de práctica U4–U5, sección II, ítem 1 (predecir la salida).
 * Se ejecuta con: mpirun --oversubscribe -np 4 ./item_II1_mpi   (y con -np 1 para el inciso d)
 */
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    int rank, size, value, total = 0, from_left = -1, max_root = 0;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    value = rank + 1;
    int right = (rank + 1) % size, left = (rank + size - 1) % size;
    MPI_Allreduce(&value, &total, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
    MPI_Sendrecv(&value, 1, MPI_INT, right, 7, &from_left, 1, MPI_INT, left, 7,
                 MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    MPI_Reduce(&from_left, &max_root, 1, MPI_INT, MPI_MAX, 0, MPI_COMM_WORLD);
    printf("rango %d: value=%d total=%d from_left=%d max_root=%d\n", rank, value, total, from_left, max_root);
    MPI_Finalize();
    return 0;
}


In [ ]:
%%writefile item_III1_mean_buggy.c
/* item_III1_mean_buggy.c — Examen de práctica U4–U5, sección III, ítem 1 (encontrar y corregir el error).
 *
 * Qué debe hacer: la raíz reparte x[8] = {1,…,8} en bloques de 2; cada rango resta la MEDIA GLOBAL (4.5)
 * a su bloque e imprime el resultado. Exactamente 4 procesos.
 * Síntoma con el error: compila, no se cuelga, el rango 0 imprime media=4.50 y los rangos 1–3 imprimen
 * media=0.00 (centran su bloque con una media que no es).
 *   mpicc -std=c11 -O2 -Wall -Wextra item_III1_mean_buggy.c -o item_III1_mean_buggy
 *   mpirun --oversubscribe -np 4 ./item_III1_mean_buggy
 * La corrección del examen se obtiene con -DCORREGIDO (la clave está en examen-practica-U4-U5.md).
 */
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    int rank, size;
    const int n = 8;
    double x[8], local[2], local_sum, total = 0.0;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (size != 4) {                              /* el reparto fijo 8 = 4 x 2 exige exactamente 4 procesos */
        if (rank == 0) fprintf(stderr, "este item requiere exactamente 4 procesos (hay %d)\n", size);
        MPI_Finalize();
        return 1;
    }
    if (rank == 0)
        for (int i = 0; i < n; i++) x[i] = i + 1.0;                        /* 1 2 3 4 5 6 7 8 */
    MPI_Scatter(x, 2, MPI_DOUBLE, local, 2, MPI_DOUBLE, 0, MPI_COMM_WORLD); /* rango r: x[2r], x[2r+1] */
    local_sum = local[0] + local[1];
#ifdef CORREGIDO
    MPI_Allreduce(&local_sum, &total, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);   /* total en TODOS */
#else
    MPI_Reduce(&local_sum, &total, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);   /* total solo en la raíz */
#endif
    double mean = total / n;                      /* cada rango usa "su" total */
    local[0] -= mean;
    local[1] -= mean;
    printf("rango %d: media=%.2f centrado=%.2f %.2f\n", rank, mean, local[0], local[1]);
    MPI_Finalize();
    return 0;
}


In [ ]:
%%writefile item_IV_dot_mpi.c
/* item_IV_dot_mpi.c — Examen de práctica U4–U5, sección IV(a), con los huecos YA llenos (clave).
 * Producto punto distribuido; n divisible entre el número de procesos.
 *   mpirun --oversubscribe -np 4 ./item_IV_dot_mpi      → dot=36.000000  (x[i] = i+1, y[i] = 1, n = 8)
 */
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
    int rank, size;
    const int n = 8;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (n % size != 0) {
        if (rank == 0) fprintf(stderr, "n=%d debe ser divisible entre %d procesos\n", n, size);
        MPI_Finalize();
        return 1;
    }
    double x[8], y[8];
    if (rank == 0)
        for (int i = 0; i < n; i++) { x[i] = i + 1.0; y[i] = 1.0; }

    int local_n = n / size;
    double *local_x = malloc(local_n * sizeof(double));
    double *local_y = malloc(local_n * sizeof(double));
    /* (1) cuántos elementos recibe CADA proceso: local_n, en el envío y en la recepción */
    MPI_Scatter(x, local_n, MPI_DOUBLE, local_x, local_n, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    MPI_Scatter(y, local_n, MPI_DOUBLE, local_y, local_n, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    double local_dot = 0.0, dot = 0.0;
    for (int i = 0; i < local_n; i++) local_dot += local_x[i] * local_y[i];
    /* (2) la colectiva que junta en la raíz; (3) tipo y operación */
    MPI_Reduce(&local_dot, &dot, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) printf("dot=%f\n", dot);
    free(local_x);
    free(local_y);
    MPI_Finalize();
    return 0;
}


In [ ]:
!mpicc -std=c11 -O2 -Wall -Wextra item_II1_mpi.c -o item_II1_mpi && mpicc -std=c11 -O2 -Wall -Wextra item_III1_mean_buggy.c -o item_III1_mean_buggy \
  && mpicc -std=c11 -O2 -Wall -Wextra -DCORREGIDO item_III1_mean_buggy.c -o item_III1_mean_fixed && mpicc -std=c11 -O2 -Wall -Wextra item_IV_dot_mpi.c -o item_IV_dot_mpi && echo 'items MPI listos'
!echo '== II.1 con 4 procesos (las lineas salen en cualquier orden) y con 1'; timeout 5 mpirun --allow-run-as-root --oversubscribe -np 4 ./item_II1_mpi; timeout 5 mpirun --allow-run-as-root --oversubscribe -np 1 ./item_II1_mpi
!echo '== III.1 con el error'; timeout 5 mpirun --allow-run-as-root --oversubscribe -np 4 ./item_III1_mean_buggy
!echo '== III.1 corregido'; timeout 5 mpirun --allow-run-as-root --oversubscribe -np 4 ./item_III1_mean_fixed
!echo '== IV.a'; timeout 5 mpirun --allow-run-as-root --oversubscribe -np 4 ./item_IV_dot_mpi

## Guardar resultados (CSV)
El archivo lleva la fecha y la GPU. Si usas estas mediciones en algún reporte, súbelo a `resultados/` de tu repositorio.

In [ ]:
import csv
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace(" ", "_")
fecha = datetime.date.today().isoformat()
nombre = f"resultados_S06_{fecha}_{gpu or 'gpu'}.csv"
with open(nombre, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["fecha", "gpu", "programa", "n", "block", "verificacion", "t_kernel_ms", "t_cpu_ms", "speedup_kernel"])
    for fila in filas:
        w.writerow([fecha, gpu] + fila)
print("guardado:", nombre)
try:
    from google.colab import files
    files.download(nombre)
except Exception:
    print("descarga manual desde el panel de archivos (no estás en Colab o el navegador bloqueó la descarga)")
